# Worked solutions — Expert practice: measurement, design, and observability

**Stage:** Expert practice · **Suggested time:** 45–90 minutes (capstone: several sessions)

**Prerequisites:** Concepts from notebooks 00–21. No variables or files from other notebooks are required.

## What you will learn

- Measure before optimizing
- Profile a repeatable workload
- Separate pure logic from I/O and inject dependencies


This notebook contains complete worked answers. Try the corresponding learning notebook first. These are possible solutions, not the only correct approaches. Run from the top so all examples and answer checks execute.


## 1. Compare equivalent work

Timing depends on hardware, interpreter, input size, and background activity. Compare equal results and include relevant setup costs. timeit repeats work; do not turn a timing ratio into a universal rule.


In [1]:
import timeit
numbers = list(range(5000))
targets = list(range(4900, 5000))

def list_lookups():
    return sum(target in numbers for target in targets)

def set_lookups():
    indexed = set(numbers)  # Include construction in this comparison.
    return sum(target in indexed for target in targets)

assert list_lookups() == set_lookups() == 100
for function in [list_lookups, set_lookups]:
    seconds = min(timeit.repeat(function, number=3, repeat=3)) / 3
    print(function.__name__, f"{seconds:.6f} seconds per call (best sample)")


list_lookups 0.003258 seconds per call (best sample)
set_lookups 0.000125 seconds per call (best sample)


**Try a change:** Try one lookup instead of 100. Does building a set still help?


## 2. Find where time goes

A profiler attributes time to functions. Use a representative workload before choosing an optimization. cProfile adds overhead, so use it to locate work; use focused timing to compare a proposed change.


In [2]:
import cProfile
import pstats
import io

def workload():
    return sum(sum(n * n for n in range(1000)) for _ in range(20))

profiler = cProfile.Profile()
profiler.runcall(workload)
report = io.StringIO()
pstats.Stats(profiler, stream=report).strip_dirs().sort_stats("cumulative").print_stats(5)
print(report.getvalue())


         44 function calls (24 primitive calls) in 0.004 seconds

   Ordered by: cumulative time

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
        1    0.000    0.000    0.006    0.006 1976220401.py:5(workload)
     21/1    0.004    0.000    0.006    0.006 {built-in method builtins.sum}
       21    0.000    0.000    0.006    0.000 1976220401.py:6(<genexpr>)
        1    0.000    0.000    0.000    0.000 {method 'disable' of '_lsprof.Profiler' objects}





**Try a change:** Replace the inner generator with another correct implementation. Check equality before timing.


## 3. Pure core, explicit boundaries

Keep calculations separate from loading and saving. Passing a loader makes the workflow testable without a real filesystem or API. Log useful events without including sensitive records. Configure an isolated logger to avoid duplicate notebook handlers.


In [3]:
import logging
import io

def summarize(records):
    return {"count": len(records), "total": sum(record["amount"] for record in records)}

def build_report(load_records, logger):
    records = load_records()
    logger.info("Loaded %d records", len(records))
    return summarize(records)

log_output = io.StringIO()
logger = logging.Logger("course_report", level=logging.INFO)
handler = logging.StreamHandler(log_output)
logger.addHandler(handler)
try:
    report = build_report(lambda: [{"amount": 10}, {"amount": 20}], logger)
    print(report)
    print(log_output.getvalue().strip())
finally:
    logger.removeHandler(handler)
    handler.close()
assert report == {"count": 2, "total": 30}


{'count': 2, 'total': 30}
Loaded 2 records


**Try a change:** Use an empty loader and verify the summary. Decide which failures belong at the loading boundary.


## Practice: solve before looking at the answers

Replace the starter code with your implementation. Run it, then change that exercise's `exercise_N_ready` flag to `True` and run its check cell. A failed assertion is feedback: compare the expected value with your result. Add at least one test of your own.


### Exercise 1: Index once

Implement index_by_id(records) returning a dict keyed by id. Raise ValueError for duplicate IDs instead of silently overwriting them. Each record has a hashable id.

<details>
<summary>Show a hint</summary>

Check membership before inserting.

</details>


In [4]:
def index_by_id(records):
    index = {}
    for record in records:
        key = record["id"]
        if key in index:
            raise ValueError(f"duplicate id: {key}")
        index[key] = record
    return index


In [5]:
exercise_1_ready = True  # Set True when your implementation is ready.
if exercise_1_ready:
    assert index_by_id([{"id": 1, "name": "A"}])[1]["name"] == "A"
    assert index_by_id([]) == {}
    try:
        index_by_id([{"id": 1}, {"id": 1}])
    except ValueError:
        pass
    else:
        raise AssertionError("duplicate IDs must fail")
    print("Exercise 1: supplied checks passed. Add your own edge cases!")
else:
    print("Exercise 1: not checked yet. Complete the code, then set exercise_1_ready = True.")


Exercise 1: supplied checks passed. Add your own edge cases!


### Exercise 2: Inject a loader

Implement total_from_loader(loader): call loader exactly once, then sum amount from its records. Return 0 for no records.

<details>
<summary>Show a hint</summary>

Keep the data source outside the calculation.

</details>


In [6]:
def total_from_loader(loader):
    records = loader()
    return sum(record["amount"] for record in records)


In [7]:
exercise_2_ready = True  # Set True when your implementation is ready.
if exercise_2_ready:
    calls = []
    def fake_loader():
        calls.append("called")
        return [{"amount": 3}, {"amount": 4}]
    assert total_from_loader(fake_loader) == 7
    assert calls == ["called"]
    assert total_from_loader(lambda: []) == 0
    print("Exercise 2: supplied checks passed. Add your own edge cases!")
else:
    print("Exercise 2: not checked yet. Complete the code, then set exercise_2_ready = True.")


Exercise 2: supplied checks passed. Add your own edge cases!


## Build it yourself

Refactor one earlier mini-project into validation, pure calculations, and I/O. Add behavior tests, a representative benchmark, and a short tradeoff note covering complexity, memory, readability, and failure handling.

This open-ended extension has no single supplied solution. Record your design choices and test your result.


In [8]:
# Your mini-project workspace. Add more cells as needed.


## Common mistakes and limits

Expertise is not a list of syntax features. Practice making justified tradeoffs, testing failure paths, reading existing code, and revisiting decisions with evidence. These notebooks are a foundation for that ongoing work.


## Before you move on

- [ ] I can explain the examples without reading the code.
- [ ] I changed at least two inputs and predicted the results.
- [ ] I completed both exercises and enabled their checks.
- [ ] I added a boundary or failure case of my own.
- [ ] I attempted the mini-project and recorded one lesson learned.

**Reflection:** What surprised you? What would you try next? Edit this Markdown cell with your notes.


[Course guide](../README.md) · [Back to learning notebook](../notebooks/22_performance_and_software_design.ipynb)
